# CIS 5270 Final Project — LLM Code Style Distillation via SFT → DPO → RFT

**Team:** Sanjana Gundapaneni, Roberto Ligeralde, Mick Yang

**Goal:** Fine-tune GPT-nano to refactor correct-but-verbose LLM-generated code into more human-like Python, without sacrificing correctness.

**Pipeline:**
1. Generate LLM solutions from EvalPlus (GPT-mini as teacher)
2. Prepare SFT / DPO / RFT datasets
3. SFT warm-start — distill human reference style into GPT-nano
4. DPO training — preference optimisation for style and correctness
5. RFT training — reinforcement fine-tuning with combined reward
6. Benchmark — compare all models on correctness and style

## 1. Setup and Installation

In [ ]:
%pip install -q -r requirements.txt

## 2. Configure Azure

In [ ]:
import os
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential
import config

RESOURCE_GROUP  = config.RESOURCE_GROUP
OPENAI_API_KEY  = config.OPENAI_API_KEY
OPENAI_ENDPOINT = config.OPENAI_ENDPOINT
SUBSCRIPTION_ID = config.SUBSCRIPTION_ID

os.environ['AZURE_SUBSCRIPTION_ID']  = SUBSCRIPTION_ID
os.environ['AZURE_RESOURCE_GROUP']   = 'CIS-5270'
os.environ['AZURE_AOAI_ACCOUNT']     = RESOURCE_GROUP
os.environ['AZURE_OPENAI_API_KEY']   = OPENAI_API_KEY
os.environ['AZURE_OPENAI_ENDPOINT']  = OPENAI_ENDPOINT

client = AzureOpenAI(
    api_key=OPENAI_API_KEY,
    azure_endpoint=OPENAI_ENDPOINT,
    api_version='2025-04-01-preview',
)
credential = DefaultAzureCredential()

print('Connected to Azure OpenAI')

## 3. Data Generation

### 3-1. Generate LLM solutions from EvalPlus

Uses GPT-mini to produce verbose and natural solutions for each HumanEval+ problem.
Correctness is verified immediately via `exec()` against EvalPlus test cases.

In [ ]:
# Generates data_files/llm_solutions.jsonl
# --n limits to N problems (omit for full dataset; ~164 HumanEval+ problems)
!python scripts/generate_solutions.py --n 20

### 3-2. Prepare SFT / DPO / RFT datasets

In [ ]:
# Generates:
#   data_files/sft_train.jsonl, sft_val.jsonl
#   data_files/dpo_train.jsonl, dpo_val.jsonl
#   data_files/rft_train.jsonl, rft_val.jsonl
!python scripts/prepare_datasets.py

In [ ]:
# sanity check: peek at one record from each dataset
import json

for fname in ['data_files/sft_train.jsonl', 'data_files/dpo_train.jsonl', 'data_files/rft_train.jsonl']:
    with open(fname) as f:
        rec = json.loads(f.readline())
    print(f'\n=== {fname} ===')
    print(json.dumps(rec, indent=2)[:600], '...')

### 3-3. Pre-compute human reference distribution

Builds `data_files/human_dist.json` — the baseline distribution of McCabe complexity,
Halstead metrics, identifier lengths, etc. from the canonical EvalPlus solutions.

In [ ]:
!python scripts/verified_rewards.py \
    --solutions data_files/llm_solutions.jsonl \
    --out       data_files/human_dist.json

## 4. SFT Warm-start

Distills the human reference style into GPT-nano via supervised fine-tuning.
The SFT model serves as the base for DPO and RFT training.

In [ ]:
import sys; sys.path.insert(0, '.')
from scripts.sft_train import run_sft

sft_model_id = run_sft(wait=True)
print(f'SFT model: {sft_model_id}')

In [ ]:
# Monitor recent events
import json

with open('data_files/sft_model_id.txt') as f:
    sft_job_model = f.read().strip()

# Retrieve latest job for this model
jobs = list(client.fine_tuning.jobs.list(limit=5))
sft_job = next((j for j in jobs if j.fine_tuned_model == sft_job_model), jobs[0])

events = list(client.fine_tuning.jobs.list_events(sft_job.id, limit=10))
for e in reversed(events):
    print(e.message)

## 5. DPO Training

Two types of preference pairs:
- **Style pairs** (chosen=human_ref, rejected=correct_LLM) — teaches human-likeness
- **Correctness pairs** (chosen=correct_LLM, rejected=incorrect_LLM) — reinforces correctness

In [ ]:
from scripts.dpo_train import run_dpo

dpo_model_id = run_dpo(wait=True)  # reads sft_model_id.txt automatically
print(f'DPO model: {dpo_model_id}')

## 6. RFT / PPO Training

Reinforcement Fine-Tuning with a custom Python grader that computes:
- **Correctness (0.7 weight):** exec solution against EvalPlus test cases  
- **Style (0.3 weight):** AST-based static analysis vs human reference
  (static analysis is the training-time surrogate — LLM-as-judge used at eval time)

The DPO model is used as the warm-started base.

In [ ]:
from scripts.rft_train import run_rft, GRADER_SOURCE
print('=== RFT Grader (Python) ===')
print(GRADER_SOURCE)

In [ ]:
rft_model_id = run_rft(wait=True)  # reads dpo_model_id.txt automatically
print(f'RFT model: {rft_model_id}')

## 7. Deploy Models for Inference

In [ ]:
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from scripts.utils import deploy_model

cogsvc_client = CognitiveServicesManagementClient(
    credential=credential, subscription_id=SUBSCRIPTION_ID
)

# Deploy the RFT model (best expected model)
deploy_model(cogsvc_client, rft_model_id, deployment_name='gpt-nano-rft')

In [ ]:
TEST_PROBLEM = '''def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check whether in given list of numbers, are any two numbers closer to each
    other than given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """
'''

VERBOSE_LLM_SOLUTION = (
    '''
        if not isinstance(numbers, list):
            raise TypeError("numbers must be a list")
        if not isinstance(threshold, float):
            raise TypeError("threshold must be a float")
        result: bool = False
        for index_i in range(len(numbers)):
            for index_j in range(len(numbers)):
                if index_i != index_j:
                    distance: float = abs(numbers[index_i] - numbers[index_j])
                    if distance < threshold:
                        result = True
        return result
    '''
)

response = client.chat.completions.create(
    model='gpt-nano-rft',
    messages=[
        {
            'role': 'system', 
            'content': 'You are an expert Python programmer who refactors verbose, LLM-generated code into clean, idiomatic Python.'},
        {
            'role': 'user', 
            'content': f'Refactor this LLM solution to be more human-like:\n\nProblem:\n{TEST_PROBLEM}\n\nLLM solution:\n```python{VERBOSE_LLM_SOLUTION}```'},
    ],
    temperature=0.0,
)
print('Refactored output:')
print(response.choices[0].message.content)

## 8. Benchmark Evaluation

Evaluates **base / SFT / DPO / RFT** models on a held-out set of EvalPlus problems.

Metrics:
- **Correctness**: pass rate on EvalPlus test cases (exec-based)
- **LLM Style**: GPT-mini-as-judge score (0–1) — primary eval signal
- **Static Style**: AST/radon metrics vs human reference distribution
- **Reward**: 0.7 × correctness + 0.3 × LLM_style

In [ ]:
# Runs inference on all deployed models and computes metrics
!python scripts/benchmark.py --n 50

In [ ]:
# Visualise results
import json

results = []
with open('data_files/benchmark_results.jsonl') as f:
    for line in f:
        results.append(json.loads(line))

print(f'{'Model':<10} {'Correct':>8} {'LLM Style':>10} {'Reward':>8}')
print('-' * 40)
for r in results:
    print(f"{r['model']:<10} {r['correctness']:>8.3f} {r['llm_style']:>10.3f} {r['reward']:>8.3f}")

## 9. Remove Deployments

In [ ]:
from scripts.utils import delete_deployment

for deployment_name in ['gpt-nano-rft', 'gpt-nano-dpo', 'gpt-nano-sft']:
    try:
        delete_deployment(cogsvc_client, deployment_name)
    except Exception as e:
        print(f'  {deployment_name}: {e}')